# Do Full-Duplex Speech Models Listen in Text? — v6

v5 установил: собственная речь модели подавляет «чтение» собеседника (PersonaPlex теряет ~70%, Moshi ~40%), подавляет именно **свой звук**, а не свой текст, подавление держится только пока модель говорит. Абляция полностью убрала чтение, но поведенческая мера не сработала: после реплики модель с вероятностью ~1 выдаёт PAD. Tuned lens на 4× данных нашёл обобщаемый текстовый код уже с 12-го слоя.

| # | Вопрос | Как проверяем |
|---|---|---|
| H10 | Что в своём звуке подавляет слушание: **слова** или **активность канала**? | Свой канал: речь (как v5) / та же речь **обращённая** / **шум со спектром речи** / речь **на −20 дБ** |
| H11 | Модель «слушает» **себя**? | Чтение слов, звучащих в своём канале, по слоям |
| H9a | Использует ли модель транскрипцию **для ответа**? | **Вынужденный ответ:** через 0,5 с после реплики в своём текстовом потоке ставим EPAD (начало слова) и смотрим распределение первого слова ответа: KL и вероятность услышанных слов при абляции своих направлений против чужих |
| H9b | Использует ли модель транскрипцию **как контекст**? | Вырезаем направления слов **первой половины** реплики (только в первой половине) и смотрим чтение слов **второй половины**; по предсказуемости слов (GPT-2); с контролем перекрытия подпространств |
| H5″ | Код с 12-го слоя — словесный или только «звук → первый кусок слова»? | Строгий тест tuned lens: слова, чей **целевой токен ни разу не встречался в обучении** |

Kaggle: GPU T4×2, Internet On, `HF_TOKEN`. ~60–80 мин на модель.

## 1. Конфигурация

In [ ]:
import os
ON_KAGGLE = os.path.exists("/kaggle/working")
ON_COLAB = "COLAB_RELEASE_TAG" in os.environ or os.path.exists("/content")
BASE = "/kaggle/working" if ON_KAGGLE else ("/content" if ON_COLAB else os.getcwd())

CFG = dict(
    model          = "personaplex",              # "personaplex" | "moshi"
    datasets       = ["librispeech", "ami"],
    conditions     = ["listen", "talking_audio", "own_reversed", "own_noise", "own_quiet"],
    quiet_gain     = 0.1,                        # -20 dB
    n_turns        = 40,
    turn_sec       = 20.0,
    gap_sec        = 0.3,
    aligner        = "mms",
    whisper_model  = "small.en",
    silence_frames = 6,
    post_sec       = 1.5,
    resp_gap_sec   = 0.5,                        # H9a: EPAD forced this long after the user stops
    topk           = 20,
    lag_frames     = 4,
    n_control      = 5,
    min_word_chars = 4,
    n_boot         = 1000,
    ab_layers      = list(range(22, 31)),
    surprisal_lm   = "gpt2",
    # H5'' tuned lens
    tl_extra       = ["dev-clean", "test-other"],
    tl_extra_turns = 60,
    tl_layers      = [0, 4, 8, 12, 16, 20, 24, 28, 32],
    tl_ranks       = [128],
    tl_steps       = 800,
    tl_batch       = 512,
    tl_lr          = 1e-3,
    tl_wd          = 1e-2,
    tl_train_frac  = 0.7,
    dtype          = "auto",
    seed           = 0,
)
PRESETS = {
    "personaplex": dict(model_repo="nvidia/personaplex-7b-v1",
                        text_prompt="You enjoy having a good conversation.", prefix_user="sine"),
    "moshi":       dict(model_repo="kyutai/moshiko-pytorch-bf16", text_prompt="", prefix_user="silence"),
}
CFG.update(PRESETS[CFG["model"]])
CFG["out_dir"] = f"{BASE}/{CFG['model']}_v6"
os.makedirs(CFG["out_dir"], exist_ok=True)
print(CFG["model_repo"], "->", CFG["out_dir"])

## 2. Установка и модель

In [ ]:
import sys, subprocess
os.environ["NO_TORCH_COMPILE"] = "1"
os.environ["NO_CUDA_GRAPH"] = "1"
if not os.path.exists("personaplex"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/NVIDIA/personaplex.git"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "einops", "sentencepiece", "sphn", "openai-whisper",
                "soundfile", "tqdm", "wordfreq", "datasets"], check=True)
sys.path.insert(0, os.path.abspath("personaplex/moshi"))

In [ ]:
import json, math, random, re, tarfile, urllib.request, gc, io, time, warnings
from pathlib import Path
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
import sentencepiece, soundfile as sf
from scipy.signal import resample_poly
from safetensors import safe_open
from moshi.models import loaders
from moshi.models.lm import LMModel, SILENCE_TOKENS, SINE_TOKENS, _delay_sequence
warnings.filterwarnings("ignore", category=RuntimeWarning)
random.seed(CFG["seed"]); np.random.seed(CFG["seed"]); torch.manual_seed(CFG["seed"])
print(torch.__version__, [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
from huggingface_hub import hf_hub_download, list_repo_files
if not os.environ.get("HF_TOKEN"):
    try:
        if ON_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
        elif ON_COLAB:
            from google.colab import userdata
            os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception as e:
        print("could not read HF_TOKEN secret:", e)
repo = CFG["model_repo"]
files = list_repo_files(repo)
lm_file = loaders.MOSHI_NAME if loaders.MOSHI_NAME in files else next(
    f for f in files if f.endswith(".safetensors") and not f.startswith("tokenizer"))
moshi_path = hf_hub_download(repo, lm_file)
mimi_path  = hf_hub_download(repo, loaders.MIMI_NAME)
sp = sentencepiece.SentencePieceProcessor(hf_hub_download(repo, loaders.TEXT_TOKENIZER_NAME))

In [ ]:
SPECIAL_TEXT = [0, 1, 2, 3]   # EPAD, BOS, EOS, PAD
SKIP_PREFIXES = ("depformer", "linears.")

def pick_dtype():
    if CFG["dtype"] != "auto":
        return {"fp16": torch.float16, "bf16": torch.bfloat16}[CFG["dtype"]]
    major = torch.cuda.get_device_capability(0)[0] if torch.cuda.is_available() else 0
    return torch.bfloat16 if major >= 8 else torch.float16

def layer_devices(n_layers):
    n_gpu = torch.cuda.device_count()
    if n_gpu == 0: return ["cpu"] * n_layers
    if n_gpu == 1: return ["cuda:0"] * n_layers
    half = n_layers // 2
    return ["cuda:0"] * half + ["cuda:1"] * (n_layers - half)

def load_lm_for_lens(path, lm_kwargs, dtype):
    with torch.device("meta"):
        lm = LMModel(device="meta", dtype=dtype, **dict(lm_kwargs))
    devs = layer_devices(len(lm.transformer.layers))
    expected = set(lm.state_dict().keys())
    def target_device(key):
        m = re.match(r"transformer\.layers\.(\d+)\.", key)
        if m: return devs[int(m.group(1))]
        if key.startswith(("emb.", "text_emb.")): return devs[0]
        return devs[-1]
    sd = {}
    with safe_open(path, framework="pt", device="cpu") as f:
        for key in f.keys():
            if key.startswith(SKIP_PREFIXES) or key not in expected:
                continue
            t = f.get_tensor(key)
            sd[key] = t.to(device=target_device(key), dtype=torch.float32 if key.endswith(".alpha") else dtype)
    missing = [k for k in expected if k not in sd and not k.startswith(SKIP_PREFIXES)]
    assert not missing, f"missing weights: {missing[:5]}"
    lm.load_state_dict(sd, strict=False, assign=True)
    lm.eval()
    for p in lm.parameters():
        p.requires_grad_(False)
    return lm, devs

DTYPE = pick_dtype()
lm, LAYER_DEVS = load_lm_for_lens(moshi_path, dict(loaders._lm_kwargs, dep_q=16), DTYPE)
N_LAYERS = len(lm.transformer.layers) + 1
mimi = loaders.get_mimi(mimi_path, device="cuda:0" if torch.cuda.is_available() else "cpu")
FRAME = int(mimi.sample_rate / mimi.frame_rate); FPS = mimi.frame_rate
print("dtype", DTYPE, "| devices", sorted(set(LAYER_DEVS)))

## 3. Данные и границы слов

In [ ]:
def norm_words(text):
    return [w for w in (re.sub(r"[^a-z']", "", t) for t in text.lower().split()) if w.strip("'")]

def librispeech_turns(n_turns, turn_sec, gap_sec, sr_out=24000, subset="test-clean"):
    root = Path(f"{BASE}/librispeech")
    if not (root / "LibriSpeech" / subset).exists():
        root.mkdir(parents=True, exist_ok=True)
        tgz = root / f"{subset}.tar.gz"
        urllib.request.urlretrieve(f"https://www.openslr.org/resources/12/{subset}.tar.gz", tgz)
        with tarfile.open(tgz) as t:
            t.extractall(root)
        tgz.unlink()
    chapters = sorted((root / "LibriSpeech" / subset).glob("*/*"))
    random.Random(CFG["seed"]).shuffle(chapters)
    turns = []
    for ch in chapters:
        trans = dict(l.split(" ", 1) for l in next(ch.glob("*.trans.txt")).read_text().splitlines())
        pieces, segs, t = [], [], 0.0
        for uid in sorted(trans):
            wav, sr = sf.read(ch / f"{uid}.flac", dtype="float32")
            wav = resample_poly(wav, sr_out // 1000, sr // 1000).astype(np.float32)
            segs.append((t, t + len(wav) / sr_out, trans[uid]))
            pieces += [wav, np.zeros(int(gap_sec * sr_out), np.float32)]
            t += len(wav) / sr_out + gap_sec
            if t >= turn_sec:
                break
        turns.append(dict(id=f"ls-{subset}-{ch.parent.name}-{ch.name}", spk=f"ls-{ch.parent.name}", wav=np.concatenate(pieces), segments=segs))
        if len(turns) >= n_turns:
            break
    return turns

def ami_turns(n_turns, turn_sec, gap_sec, sr_out=24000, max_items=20000):
    from datasets import load_dataset, Audio
    ds = load_dataset("edinburghcstr/ami", "ihm", split="test", streaming=True)
    ds = ds.cast_column("audio", Audio(decode=False))
    by_spk = {}
    for k, ex in enumerate(ds):
        if k >= max_items: break
        key = (ex["meeting_id"], ex["speaker_id"])
        by_spk.setdefault(key, []).append(ex)
    turns = []
    for (meeting, spk), exs in sorted(by_spk.items()):
        exs.sort(key=lambda e: e["begin_time"])
        pieces, segs, t = [], [], 0.0
        for ex in exs:
            if len(norm_words(ex["text"])) < 3:
                continue
            a = ex["audio"]
            wav, sr = sf.read(io.BytesIO(a["bytes"]) if a.get("bytes") else a["path"], dtype="float32")
            if wav.ndim > 1: wav = wav.mean(1)
            wav = resample_poly(wav, sr_out, sr).astype(np.float32)
            segs.append((t, t + len(wav) / sr_out, ex["text"]))
            pieces += [wav, np.zeros(int(gap_sec * sr_out), np.float32)]
            t += len(wav) / sr_out + gap_sec
            if t >= turn_sec:
                turns.append(dict(id=f"ami-{meeting}-{spk}-{len(turns)}", spk=f"ami-{spk}", wav=np.concatenate(pieces), segments=segs))
                pieces, segs, t = [], [], 0.0
                if sum(1 for x in turns if x["spk"] == f"ami-{spk}") >= 3:
                    break                                    # at most 3 turns per speaker for diversity
        if len(turns) >= n_turns:
            break
    return turns

DATA = {}
for name in CFG["datasets"]:
    try:
        DATA[name] = (librispeech_turns if name == "librispeech" else ami_turns)(CFG["n_turns"], CFG["turn_sec"], CFG["gap_sec"])
        print(f"{name}: {len(DATA[name])} turns, {sum(len(t['wav']) for t in DATA[name]) / 24000 / 60:.1f} min, "
              f"{len({t['spk'] for t in DATA[name]})} speakers")
    except Exception as e:
        print(f"!! {name} skipped: {type(e).__name__}: {e}")
assert DATA, "no dataset loaded"
MAIN = [n for n in CFG["datasets"] if n in DATA]
for sub in CFG["tl_extra"]:
    try:
        DATA[f"ls_{sub}"] = librispeech_turns(CFG["tl_extra_turns"], CFG["turn_sec"], CFG["gap_sec"], subset=sub)
        print(f"ls_{sub}: {len(DATA[f'ls_{sub}'])} turns (tuned lens only)")
    except Exception as e:
        print(f"!! ls_{sub} skipped: {type(e).__name__}: {e}")

In [ ]:
def whisper_align(turns):
    import whisper
    free = torch.cuda.mem_get_info(0)[0] / 1e9 if torch.cuda.is_available() else 0
    dev = "cuda:0" if free > 3 else "cpu"
    wm = whisper.load_model(CFG["whisper_model"], device=dev)
    for t in turns:
        res = wm.transcribe(resample_poly(t["wav"], 2, 3).astype(np.float32), language="en",
                            word_timestamps=True, condition_on_previous_text=False, fp16=(dev != "cpu"))
        t["words"] = [dict(word=re.sub(r"[^a-z']", "", w["word"].lower()), start=float(w["start"]), end=float(w["end"]))
                      for s in res["segments"] for w in s.get("words", [])]
        t["words"] = [w for w in t["words"] if w["word"].strip("'")]
    del wm; gc.collect(); torch.cuda.empty_cache()

def mms_align(turns):
    import torchaudio
    bundle = torchaudio.pipelines.MMS_FA
    dev = "cuda:0" if torch.cuda.is_available() else "cpu"
    model = bundle.get_model(with_star=False).to(dev).eval()
    tok, aligner = bundle.get_tokenizer(), bundle.get_aligner()
    for t in turns:
        words = []
        for t0, t1, text in t["segments"]:
            ws = norm_words(text)
            if not ws:
                continue
            seg = t["wav"][int(t0 * 24000):int(t1 * 24000)]
            wav16 = torch.from_numpy(resample_poly(seg, 2, 3).astype(np.float32))[None].to(dev)
            with torch.inference_mode():
                emission, _ = model(wav16)
            spans = aligner(emission[0], tok(ws))
            ratio = wav16.shape[1] / emission.shape[1] / 16000
            words += [dict(word=w, start=t0 + s[0].start * ratio, end=t0 + s[-1].end * ratio) for w, s in zip(ws, spans)]
        t["words"] = words
    del model; gc.collect(); torch.cuda.empty_cache()

ALIGNER_USED = CFG["aligner"]
for name, turns in DATA.items():
    try:
        if CFG["aligner"] != "mms": raise RuntimeError("whisper requested")
        mms_align(turns)
    except Exception as e:
        print(f"!! MMS alignment failed for {name} ({type(e).__name__}: {e}); falling back to Whisper")
        ALIGNER_USED = "whisper"
        whisper_align(turns)
    print(name, sum(len(t["words"]) for t in turns), "words; e.g.", turns[0]["words"][:4])
CFG["aligner_used"] = ALIGNER_USED

## 4. Варианты своего канала (H10)
Для реплики `other`, которую модель «произносит» поверх собеседника, кодируем четыре версии звука: исходную речь, ту же речь задом наперёд (тот же голос и спектр, нет слов), шум с тем же долговременным спектром (случайные фазы: нет ни слов, ни ритма речи) и речь на −20 дБ. Свой текстовый поток во всех вариантах — PAD.

In [ ]:
@torch.no_grad()
def mimi_codes(wav):
    pad = (-len(wav)) % FRAME
    x = torch.from_numpy(np.pad(wav, (0, pad))).to(next(mimi.parameters()).device).view(1, 1, -1)
    return mimi.encode(x)[0].cpu()

def speech_shaped_noise(wav, seed):
    X = np.fft.rfft(wav)
    ph = np.random.default_rng(seed).uniform(0, 2 * np.pi, len(X))
    y = np.fft.irfft(np.abs(X) * np.exp(1j * ph), n=len(wav)).astype(np.float32)
    return y * (np.sqrt(np.mean(wav ** 2)) / (np.sqrt(np.mean(y ** 2)) + 1e-9))

def wrap_system(text):
    return f"<system> {text.strip()} <system>"

SIL = torch.as_tensor(SILENCE_TOKENS).view(8, 1)
OWN_SRC = {"talking_audio": "codes_user", "own_reversed": "codes_rev", "own_noise": "codes_noise", "own_quiet": "codes_quiet"}

def own_streams(other, n, cond):
    codes = SIL.repeat(1, n).clone()
    text = np.full(n, 3, dtype=np.int64)
    if cond == "listen":
        return codes, text, (0, 0)
    src = other[OWN_SRC[cond]]
    L = min(n, src.shape[1])
    codes[:, :L] = src[:, :L]
    return codes, text, (0, L)

def build_codes(user_codes, own_codes, own_text, n_post):
    pad = 3
    filler = torch.as_tensor(SINE_TOKENS).view(8, 1) if CFG["prefix_user"] == "sine" else SIL
    cols = []
    def frame(tt, ma, ua): cols.append(torch.cat([torch.tensor([[int(tt)]]), ma, ua], 0))
    for _ in range(CFG["silence_frames"]): frame(pad, SIL, filler)
    for tok in PROMPT_IDS: frame(tok, SIL, filler)
    for _ in range(CFG["silence_frames"]): frame(pad, SIL, filler)
    start = len(cols)
    for i in range(user_codes.shape[1]):
        frame(own_text[i], own_codes[:, i:i + 1], user_codes[:, i:i + 1])
    for _ in range(n_post): frame(pad, SIL, SIL)
    return torch.cat(cols, 1).long(), start

PROMPT_IDS = sp.encode(wrap_system(CFG["text_prompt"])) if CFG["text_prompt"] else []
N_POST = int(round(CFG["post_sec"] * FPS))
for name, turns in DATA.items():
    for i, t in enumerate(turns):
        t["codes_user"] = mimi_codes(t["wav"])
        if name in MAIN:
            t["codes_rev"] = mimi_codes(np.ascontiguousarray(t["wav"][::-1]))
            t["codes_noise"] = mimi_codes(speech_shaped_noise(t["wav"], CFG["seed"] + i))
            t["codes_quiet"] = mimi_codes(t["wav"] * CFG["quiet_gain"])
print("ok, post frames:", N_POST)

## 5. Прямой проход

In [ ]:
@torch.no_grad()
def lens_forward(codes, topk, keep_pos=None, keep_layers=(), ablate=None, final_pos=None, check_layer=None):
    # ablate: (Q dict device->[D,k], a, b): remove span(Q) from the residual at positions [a, b) after layers in ab_layers
    first, last = LAYER_DEVS[0], LAYER_DEVS[-1]
    codes = codes.unsqueeze(0).to(first)
    initial = lm._get_initial_token().to(first)
    seq = torch.cat([initial, _delay_sequence(lm.delays, codes, initial)], 2)[:, :, :-1]
    x = lm.embed_codes(seq)
    special = torch.tensor(SPECIAL_TEXT, device=last)
    def lens(h):
        logits = lm.text_linear(lm.out_norm(h.to(last))).float()[0]
        logits[:, special] = -float("inf")
        return logits.topk(topk, -1).indices.int().cpu()
    tops, kept, check = [], {}, None
    def keep(L, h):
        if keep_pos is not None and L in keep_layers:
            kept[L] = h[0, keep_pos].to(torch.float16).cpu()
    if ablate is None:
        tops.append(lens(x)); keep(0, x)
    for i, layer in enumerate(lm.transformer.layers):
        x = layer(x.to(LAYER_DEVS[i]))
        if ablate is not None and (i + 1) in CFG["ab_layers"]:
            Q, a, b = ablate
            q = Q[str(x.device)]
            seg = x[0, a:b].float()
            x[0, a:b] = (seg - (seg @ q) @ q.T).to(x.dtype)
        if not torch.isfinite(x).all():
            raise FloatingPointError(f"non-finite activations after layer {i}")
        if ablate is None:
            tops.append(lens(x)); keep(i + 1, x)
        elif check_layer == i + 1:
            check = lens(x)[:, :5]
    final = None
    if final_pos is not None:
        final = torch.log_softmax(lm.text_linear(lm.out_norm(x.to(last))).float()[0, final_pos], -1).cpu()
    if ablate is not None:
        return check, final
    return torch.stack(tops), kept, final

def word_window(t, w, lag):
    f0 = int(math.floor(w["start"] * FPS)); f1 = int(math.ceil(w["end"] * FPS))
    return t["user_start"] + f0 + 1, min(t["user_start"] + f1 + 1 + lag, t["T"] - 1)


RUNS, TL_FRAMES = {}, {}
t0 = time.time()
for name, turns in DATA.items():
    conds = CFG["conditions"] if name in MAIN else ["listen"]
    for cond in conds:
        out = []
        for ti, t in enumerate(turns):
            other = turns[(ti + 1) % len(turns)]
            n = t["codes_user"].shape[1]
            own_codes, own_text, interval = own_streams(other, n, cond)
            codes, start = build_codes(t["codes_user"], own_codes, own_text, N_POST)
            # words that actually sound in the model's own channel (H11): only for the forward-speech variants
            own_words = [w for w in other["words"] if w["end"] * FPS < interval[1]] if cond in ("talking_audio", "own_quiet") else []
            rt = dict(id=t["id"], spk=t["spk"], user_start=start, n_user=n, T=codes.shape[1], words=t["words"],
                      text=" ".join(w["word"] for w in t["words"]), own_interval=interval, own_ids=[],
                      own_words=own_words, codes=codes if cond == "listen" else None)
            keep_pos, meta = None, []
            if cond == "listen":
                pos = []
                for wi, w in enumerate(t["words"]):
                    a, b = word_window(rt, w, 0)
                    for p in range(a, b + 1):
                        pos.append(p); meta.append((ti, wi, p - a))
                keep_pos = torch.tensor(pos, dtype=torch.long)
            rt["top"], kept, _ = lens_forward(codes, CFG["topk"], keep_pos, CFG["tl_layers"] if cond == "listen" else ())
            if cond == "listen":
                TL_FRAMES.setdefault(name, dict(meta=[], h={L: [] for L in CFG["tl_layers"]}))
                TL_FRAMES[name]["meta"] += meta
                for L, h in kept.items():
                    TL_FRAMES[name]["h"][L].append(h)
            out.append(rt)
        RUNS[(name, cond)] = out
        print(f"{name}/{cond}: {len(out)} turns ({time.time() - t0:.0f}s)")
for name in TL_FRAMES:
    TL_FRAMES[name]["h"] = {L: torch.cat(v) for L, v in TL_FRAMES[name]["h"].items()}

## 6. H10: что в своём канале подавляет слушание
Парное сравнение с listen для тех же слов на лучшем слое: доля потерянного чтения для каждого варианта своего канала.

In [ ]:
def word_targets(word):
    ids = set()
    for v in {word, word.capitalize(), word.upper()}:
        enc = sp.encode(v)
        if enc: ids.add(enc[0])
    return ids - set(SPECIAL_TEXT)

def best_rank(top, a, b, targets, excl=()):
    if b < a or not targets: return None
    win = top[:, a:b + 1, :]
    hit = torch.isin(win, torch.tensor(sorted(targets), dtype=win.dtype))
    K = top.shape[2]
    if excl:
        valid = ~torch.isin(win, torch.tensor(sorted(excl), dtype=win.dtype))
        rank = valid.long().cumsum(-1) - 1
        ranks = torch.where(hit & valid, rank, torch.full_like(rank, K))
    else:
        ranks = torch.where(hit, torch.arange(K).view(1, 1, K).expand_as(win), torch.full_like(win, K))
    return ranks.amin(dim=(1, 2)).numpy()


boot_rng = np.random.default_rng(CFG["seed"])
def boot_ci(x, n=None):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    if len(x) < 10: return np.nan, np.nan
    idx = boot_rng.integers(0, len(x), (n or CFG["n_boot"], len(x)))
    m = x[idx].mean(1)
    return np.percentile(m, 2.5), np.percentile(m, 97.5)

def score_all(name, cond, which="user", seed=0):
    turns = RUNS[(name, cond)]
    rng = random.Random(seed)
    rows = []
    for ti, t in enumerate(turns):
        others = [u for j, u in enumerate(turns) if j != ti]
        words = t["words"] if which == "user" else t["own_words"]
        for wi, w in enumerate(words):
            targets = word_targets(w["word"])
            a, b = word_window(t, w, CFG["lag_frames"])
            r = best_rank(t["top"], a, b, targets)
            if r is None: continue
            c = []
            for _ in range(CFG["n_control"]):
                u = rng.choice(others)
                if w["word"] in u["text"].split(): continue
                n = b - a; lo, hi = u["user_start"] + 1, u["user_start"] + u["n_user"] - n
                if hi <= lo: continue
                s = rng.randint(lo, hi)
                c.append(best_rank(u["top"], s, s + n, targets))
            if c:
                rows.append(dict(ti=ti, wi=wi, word=w["word"], n_chars=len(w["word"]), raw=r, c_raw=np.stack(c)))
    return rows

def excess_curve(rows, min_chars):
    sel = [r for r in rows if r["n_chars"] >= min_chars]
    real = np.stack([r["raw"] < 5 for r in sel]).astype(float)
    ctrl = np.stack([(r["c_raw"] < 5).mean(0) for r in sel])
    out = []
    for L in range(real.shape[1]):
        lo, hi = boot_ci(real[:, L] - ctrl[:, L])
        out.append(dict(layer=L, real=real[:, L].mean(), control=ctrl[:, L].mean(), excess=(real[:, L] - ctrl[:, L]).mean(),
                        ci_lo=lo, ci_hi=hi, n_words=len(sel)))
    return pd.DataFrame(out)

SCORES = {(n, c): score_all(n, c, "user", CFG["seed"]) for n in MAIN for c in CFG["conditions"]}
curves, paired = [], []
for (name, cond), rows in SCORES.items():
    curves.append(excess_curve(rows, CFG["min_word_chars"]).assign(dataset=name, cond=cond))
CURVES = pd.concat(curves, ignore_index=True)
BEST = {n: int(CURVES[(CURVES.dataset == n) & (CURVES.cond == "listen")].sort_values("excess").iloc[-1].layer) for n in MAIN}
for name in MAIN:
    base = {(r["ti"], r["wi"]): r for r in SCORES[(name, "listen")] if r["n_chars"] >= CFG["min_word_chars"]}
    for cond in CFG["conditions"][1:]:
        pairs = [(base[(r["ti"], r["wi"])], r) for r in SCORES[(name, cond)] if (r["ti"], r["wi"]) in base]
        for L in range(N_LAYERS):
            lis = np.array([b_["raw"][L] < 5 for b_, r in pairs], float)
            d = lis - np.array([r["raw"][L] < 5 for b_, r in pairs], float)
            lo, hi = boot_ci(d)
            paired.append(dict(dataset=name, cond=cond, layer=L, listen=lis.mean(), drop=d.mean(), ci_lo=lo, ci_hi=hi, n_words=len(pairs)))
PAIRED = pd.DataFrame(paired)
CURVES.to_csv(f"{CFG['out_dir']}/hit_curves_v6.csv", index=False); PAIRED.to_csv(f"{CFG['out_dir']}/paired_drop_v6.csv", index=False)
print("best listen layer:", BEST)
for name in MAIN:
    p = PAIRED[(PAIRED.dataset == name) & (PAIRED.layer == BEST[name])].set_index("cond")
    p = p.assign(share_lost=p["drop"] / p.listen, share_lo=p.ci_lo / p.listen, share_hi=p.ci_hi / p.listen)
    print(f"\n{name}, layer {BEST[name]}:"); print(p[["listen", "drop", "share_lost", "share_lo", "share_hi", "n_words"]].round(3).to_string())

## 7. H11: слышит ли модель себя?
Чтение слов, звучащих в собственном канале модели (`talking_audio`, `own_quiet`), по слоям — по той же метрике, что и для собеседника.

In [ ]:
own_curves = []
for name in MAIN:
    for cond in ("talking_audio", "own_quiet"):
        rows = score_all(name, cond, "own", CFG["seed"])
        if rows:
            own_curves.append(excess_curve(rows, CFG["min_word_chars"]).assign(dataset=name, cond=cond))
OWNC = pd.concat(own_curves, ignore_index=True) if own_curves else pd.DataFrame()
OWNC.to_csv(f"{CFG['out_dir']}/own_channel_readout.csv", index=False)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
cc = {"listen": "C0", "talking_audio": "C1", "own_reversed": "C7", "own_noise": "C5", "own_quiet": "C2"}
for ax, name in zip(axes, MAIN):
    for cond in CFG["conditions"]:
        d = CURVES[(CURVES.dataset == name) & (CURVES.cond == cond)]
        ax.plot(d.layer, d.excess, color=cc[cond], lw=2, label=f"user words: {cond}")
    for cond in ("talking_audio", "own_quiet"):
        d = OWNC[(OWNC.dataset == name) & (OWNC.cond == cond)] if len(OWNC) else []
        if len(d):
            ax.plot(d.layer, d.excess, color=cc[cond], lw=1.5, ls="--", label=f"OWN-channel words: {cond}")
    ax.axhline(0, color="k", lw=.8); ax.grid(alpha=.3); ax.legend(fontsize=7)
    ax.set(title=f"{CFG['model']} / {name}", xlabel="layer", ylabel="hit@5 excess over control")
plt.tight_layout(); plt.savefig(f"{CFG['out_dir']}/own_channel_v6.png", dpi=150); plt.show()
for name in MAIN:
    if len(OWNC):
        d = OWNC[(OWNC.dataset == name)].sort_values("excess").groupby("cond").tail(1)
        print(name, "max own-channel readout:", d[["cond", "layer", "excess", "ci_lo", "ci_hi"]].round(3).to_dict("records"))

## 8. H9a: вынужденный ответ
Через `resp_gap_sec` после конца реплики в собственный текстовый поток модели ставим EPAD («сейчас начнётся слово»). Распределение текстовой головы на следующем кадре — это первое слово ответа. Сравниваем чистый прогон с двумя абляциями (на слоях 22–30, в кадрах речи собеседника): направления слов **этой** реплики против того же числа направлений слов **других** реплик.

In [ ]:
ALPHA = lm.out_norm.alpha.detach().float().view(-1).to(LAYER_DEVS[-1])
W_OUT = lm.text_linear.weight

def basis(ids):
    W = (W_OUT[torch.tensor(sorted(ids), device=W_OUT.device)].float() * ALPHA.to(W_OUT.device)).T
    Q, _ = torch.linalg.qr(W)
    return {d: Q.to(d) for d in sorted(set(LAYER_DEVS))}

def turn_ids(t):
    ids = set()
    for w in t["words"]:
        if len(w["word"]) >= CFG["min_word_chars"]:
            ids |= word_targets(w["word"])
    return ids

def read_hits(check, t):
    # share of the turn's content words readable (top-5) at check_layer, word window as in §7
    hits = []
    for w in t["words"]:
        if len(w["word"]) < CFG["min_word_chars"]: continue
        a, b = word_window(t, w, CFG["lag_frames"])
        if b < a: continue
        tg = torch.tensor(sorted(word_targets(w["word"])), dtype=check.dtype)
        hits.append(bool(torch.isin(check[a:b + 1], tg).any()))
    return np.mean(hits) if hits else np.nan


GAP = int(round(CFG["resp_gap_sec"] * FPS))
rng_ab = random.Random(CFG["seed"])
def content_ids(words):
    ids = set()
    for w in words:
        if len(w["word"]) >= CFG["min_word_chars"]:
            ids |= word_targets(w["word"])
    return ids

resp = []
t0 = time.time()
for name in MAIN:
    turns = RUNS[(name, "listen")]
    pool_all = [content_ids(u["words"]) for u in turns]
    for ti, t in enumerate(turns):
        own = pool_all[ti]
        pool = set().union(*[p for j, p in enumerate(pool_all) if j != ti]) - own
        if len(own) < 3 or len(pool) < len(own): continue
        ctrl = set(rng_ab.sample(sorted(pool), len(own)))
        a, b = t["user_start"], t["user_start"] + t["n_user"]
        e = b + GAP
        codes = t["codes"].clone(); codes[0, e] = 0              # EPAD forced in the model's own text stream
        pos = list(range(b, e + 2))
        _, _, clean = lens_forward(codes, 5, final_pos=pos)
        row = dict(dataset=name, ti=ti, k=len(own))
        own_t, ctrl_t = torch.tensor(sorted(own)), torch.tensor(sorted(ctrl))
        def summ(lp, tag):
            first = lp[-1]                                       # distribution of the first word of the answer
            row[f"p_epad_{tag}"] = lp[:-1, 0].exp().mean().item()   # turn-taking: P(EPAD) before the forced onset
            row[f"heard_{tag}"] = first.exp()[own_t].sum().item()
            row[f"other_{tag}"] = first.exp()[ctrl_t].sum().item()
            row[f"top1_{tag}"] = int(first.argmax())
            return first
        f_clean = summ(clean, "clean")
        for tag, ids in (("own", own), ("ctrl", ctrl)):
            _, fin = lens_forward(codes, 5, ablate=(basis(ids), a, b), final_pos=pos)
            f = summ(fin, tag)
            row[f"kl_{tag}"] = torch.sum(f_clean.exp() * (f_clean - f)).item()
        resp.append(row)
    print(f"{name}: forced-response ablation done ({time.time() - t0:.0f}s)")
RESP = pd.DataFrame(resp); RESP.to_csv(f"{CFG['out_dir']}/forced_response.csv", index=False)

for name, g in RESP.groupby("dataset"):
    print(f"\n=== {name}: n={len(g)} turns, ~{g.k.mean():.0f} directions ===")
    print(f"  clean: P(heard words) in first answer word {g.heard_clean.mean():.4f} | P(other words) {g.other_clean.mean():.4f} "
          f"| P(EPAD) before onset {g.p_epad_clean.mean():.4f}")
    print("  first answer words (clean, top-1):", ", ".join(sp.id_to_piece(int(i)) for i in g.top1_clean.value_counts().index[:15]))
    d = g.kl_own - g.kl_ctrl; lo, hi = boot_ci(d.values)
    print(f"  KL clean->ablated: own {g.kl_own.mean():.4f} | ctrl {g.kl_ctrl.mean():.4f} | own-ctrl {d.mean():+.4f} [{lo:+.4f}, {hi:+.4f}] | own>ctrl in {np.mean(d > 0):.0%} of turns")
    for m in ("heard", "p_epad"):
        d = (g[f"{m}_own"] - g[f"{m}_clean"]) - (g[f"{m}_ctrl"] - g[f"{m}_clean"]); lo, hi = boot_ci(d.values)
        print(f"  {m:7s} Δown {np.mean(g[f'{m}_own'] - g[f'{m}_clean']):+.5f} | Δctrl {np.mean(g[f'{m}_ctrl'] - g[f'{m}_clean']):+.5f} | own-ctrl {d.mean():+.5f} [{lo:+.5f}, {hi:+.5f}]")
    r = np.log((g.heard_own + 1e-9) / (g.heard_clean + 1e-9)) - np.log((g.heard_ctrl + 1e-9) / (g.heard_clean + 1e-9))
    lo, hi = boot_ci(r.values)
    print(f"  log-ratio of P(heard words), own vs ctrl ablation: {r.mean():+.3f} [{lo:+.3f}, {hi:+.3f}]  (<0: the answer forgets what was heard)")

## 9. H9b: транскрипция как контекст
В каждой реплике: слова первой половины (до середины) и второй (начинаются хотя бы через 160 мс после середины). Вырезаем направления слов **первой половины** только в кадрах первой половины (слои 22–30) и смотрим чтение слов **второй половины** на лучшем слое. Контроль — столько же направлений чужих слов. Чтобы исключить прямое перекрытие подпространств, для каждого слова второй половины считаем, какая доля его вектора лежит в вырезанном подпространстве (`e_own`, `e_ctrl`), и показываем результат отдельно для слов с малым перекрытием. Отдельно — по предсказуемости слов (сюрприз GPT-2).

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
lm_tok = AutoTokenizer.from_pretrained(CFG["surprisal_lm"])
lm_dev = "cuda:0" if torch.cuda.is_available() else "cpu"
slm = AutoModelForCausalLM.from_pretrained(CFG["surprisal_lm"]).to(lm_dev).eval()

@torch.no_grad()
def word_surprisal(words):
    text = " ".join(words)
    enc = lm_tok(text, return_offsets_mapping=True, return_tensors="pt")
    ids = torch.cat([torch.tensor([[lm_tok.bos_token_id]]), enc.input_ids], 1).to(lm_dev)[:, :1024]
    logp = torch.log_softmax(slm(ids).logits[0, :-1].float(), -1)
    tok_s = -logp.gather(1, ids[0, 1:, None])[:, 0].cpu().numpy() / np.log(2)
    offs = enc.offset_mapping[0].numpy()[:len(tok_s)]
    out, pos = [], 0
    for w in words:
        s, e = pos, pos + len(w); pos = e + 1
        m = (offs[:, 0] < e) & (offs[:, 1] > s)
        out.append(float(tok_s[m].sum()) if m.any() else np.nan)
    return out


def first_target_vec(word):
    enc = sp.encode(word)
    return (W_OUT[enc[0]].float() * ALPHA.to(W_OUT.device)) if enc else None

def energy(Qd, v):
    q = Qd[str(v.device)] if str(v.device) in Qd else next(iter(Qd.values())).to(v.device)
    return float(((q.T @ v) ** 2).sum() / (v @ v))

ctx = []
t0 = time.time()
for name in MAIN:
    turns = RUNS[(name, "listen")]
    L = BEST[name]
    pool_all = [content_ids(u["words"]) for u in turns]
    for ti, t in enumerate(turns):
        mid = t["n_user"] // 2
        first = [w for w in t["words"] if math.ceil(w["end"] * FPS) <= mid]
        second = [(wi, w) for wi, w in enumerate(t["words"]) if math.floor(w["start"] * FPS) >= mid + 2 and len(w["word"]) >= CFG["min_word_chars"]]
        ids_first = content_ids(first) - content_ids([w for _, w in second])
        pool = set().union(*[p for j, p in enumerate(pool_all) if j != ti]) - pool_all[ti]
        if len(ids_first) < 3 or len(second) < 3 or len(pool) < len(ids_first): continue
        ids_ctrl = set(rng_ab.sample(sorted(pool), len(ids_first)))
        Qo, Qc = basis(ids_first), basis(ids_ctrl)
        a = t["user_start"]
        chk_own, _ = lens_forward(t["codes"], 5, ablate=(Qo, a, a + mid + 1), check_layer=L)
        chk_ctrl, _ = lens_forward(t["codes"], 5, ablate=(Qc, a, a + mid + 1), check_layer=L)
        surp = word_surprisal([w["word"] for w in t["words"]])
        for wi, w in second:
            tg = torch.tensor(sorted(word_targets(w["word"])), dtype=torch.int32)
            if len(tg) == 0: continue
            s0, s1 = word_window(t, w, CFG["lag_frames"])
            hit = lambda top5: float(torch.isin(top5[s0:s1 + 1], tg).any())
            v = first_target_vec(w["word"])
            ctx.append(dict(dataset=name, ti=ti, wi=wi, word=w["word"], surprisal=surp[wi],
                            clean=hit(t["top"][L, :, :5]), own=hit(chk_own), ctrl=hit(chk_ctrl),
                            e_own=energy(Qo, v), e_ctrl=energy(Qc, v)))
    print(f"{name}: context ablation done ({time.time() - t0:.0f}s)")
CTX = pd.DataFrame(ctx).dropna(); CTX.to_csv(f"{CFG['out_dir']}/context_ablation.csv", index=False)
del slm; gc.collect(); torch.cuda.empty_cache()

def ctx_report(g, label):
    d = (g.own - g.clean) - (g.ctrl - g.clean); lo, hi = boot_ci(d.values)
    print(f"  {label:34s} n={len(g):4d} clean {g.clean.mean():.3f} | own {g.own.mean():.3f} | ctrl {g.ctrl.mean():.3f} | own-ctrl {d.mean():+.3f} [{lo:+.3f}, {hi:+.3f}]")
for name, g in CTX.groupby("dataset"):
    print(f"\n=== {name}: readout of second-half words after ablating first-half words ===")
    print(f"  subspace overlap: e_own {g.e_own.mean():.4f} vs e_ctrl {g.e_ctrl.mean():.4f}")
    ctx_report(g, "all words")
    thr = g.e_ctrl.quantile(0.75)
    ctx_report(g[g.e_own <= thr], "low overlap (e_own <= ctrl 75th pct)")
    g = g.assign(tert=pd.qcut(g.surprisal, 3, labels=["predictable", "medium", "surprising"]))
    for tert, h in g.groupby("tert", observed=True):
        ctx_report(h, f"surprisal: {tert}")

## 10. H5″: строгий tuned lens
Как в v5 (обучение на dev-clean + test-other + 70% дикторов test-clean), плюс тест **unseen_target**: слова, чей первый токен ни разу не был целевым при обучении. Если линза читает и их, код с 12-го слоя — не просто запомненное соответствие «начало звучания → кусок слова» для знакомых кусков.

In [ ]:
class LowRankLens(torch.nn.Module):
    def __init__(self, d, r):
        super().__init__()
        self.U = torch.nn.Parameter(torch.randn(d, r) * 0.01)
        self.V = torch.nn.Parameter(torch.zeros(r, d))
        self.b = torch.nn.Parameter(torch.zeros(d))
    def forward(self, h):
        # work on the RMS-normalised state: residual norms differ by orders of magnitude across layers,
        # and the model's out_norm is scale-invariant, so at init this is exactly the logit lens
        h = h * torch.rsqrt(h.pow(2).mean(-1, keepdim=True) + 1e-6)
        return h + (h @ self.U) @ self.V + self.b

LAST = LAYER_DEVS[-1]
SPECIAL_MASK = torch.zeros(lm.text_linear.out_features, dtype=torch.bool, device=LAST)
SPECIAL_MASK[SPECIAL_TEXT] = True

def lens_logits(h):
    logits = lm.text_linear(lm.out_norm(h.to(DTYPE).unsqueeze(0)))[0].float()
    return logits.masked_fill(SPECIAL_MASK, -1e4)

def frame_table(name):
    turns = RUNS[(name, "listen")]
    meta = TL_FRAMES[name]["meta"]
    rows = []
    for fi, (ti, wi, off) in enumerate(meta):
        w = turns[ti]["words"][wi]["word"]
        enc = sp.encode(w)
        rows.append(dict(fi=fi, ti=ti, wi=wi, spk=turns[ti]["spk"], word=w, target=enc[0] if enc else -1))
    return pd.DataFrame(rows)

def train_lens(H, y, steps, seed, rank, shuffle=False):
    g = torch.Generator().manual_seed(seed)
    if shuffle: y = y[torch.randperm(len(y), generator=g)]
    lens = LowRankLens(H.shape[1], rank).to(LAST)
    opt = torch.optim.AdamW(lens.parameters(), lr=CFG["tl_lr"], weight_decay=CFG["tl_wd"])
    for step in range(steps):
        idx = torch.randint(0, len(y), (CFG["tl_batch"],), generator=g)
        h = H[idx].to(LAST).float()
        loss = torch.nn.functional.cross_entropy(lens_logits(lens(h)), y[idx].to(LAST))
        opt.zero_grad(); loss.backward(); opt.step()
    return lens

@torch.no_grad()
def word_hit(lens, H, ft, k=5):
    # fraction of words (grouped by ti, wi) with target in top-k on any frame
    if len(ft) == 0:
        return np.array([])
    hits = []
    for i in range(0, len(ft), 2048):
        h = H[torch.as_tensor(ft.fi.values[i:i + 2048])].to(LAST).float()
        h = lens(h) if lens is not None else h
        top = lens_logits(h).topk(k, -1).indices.cpu()
        hits.append((top == torch.tensor(ft.target.values[i:i + 2048]).view(-1, 1)).any(-1))
    f = ft.assign(hit=torch.cat(hits).numpy())
    return f.groupby(["ti", "wi"]).hit.max().values

In [ ]:
def prep(name):
    f = frame_table(name)
    f = f[(f.target >= 0) & (f.word.str.len() >= CFG["min_word_chars"])].copy()
    f["src"] = name
    return f

FT = prep("librispeech")
spks = sorted(FT.spk.unique()); random.Random(CFG["seed"]).shuffle(spks)
train_spk = set(spks[:int(len(spks) * CFG["tl_train_frac"])])
train_sets = [("librispeech", FT[FT.spk.isin(train_spk)])] + [(n, prep(n)) for n in TL_FRAMES if n.startswith("ls_")]
test = FT[~FT.spk.isin(train_spk)]
seen = set().union(*[set(f.word) for _, f in train_sets])
seen_targets = set(np.concatenate([f.target.values for _, f in train_sets]).tolist())
tests = {"seen": test[test.word.isin(seen)], "unseen": test[~test.word.isin(seen)],
         "unseen_target": test[~test.target.isin(seen_targets)]}
if "ami" in TL_FRAMES:
    fa = prep("ami"); tests["ami_unseen"] = fa[~fa.word.isin(seen)]; tests["ami_unseen_target"] = fa[~fa.target.isin(seen_targets)]
print("train frames:", sum(len(f) for _, f in train_sets), "| word types seen in training:", len(seen),
      "| test words:", {k: v.groupby(["ti", "wi"]).ngroups for k, v in tests.items()})

def mean_ci(v):
    v = np.asarray(v, float)
    lo, hi = boot_ci(v, 300) if len(v) else (np.nan, np.nan)
    return (v.mean() if len(v) else np.nan), lo, hi

tl_rows = []
for L in CFG["tl_layers"]:
    Htr = torch.cat([TL_FRAMES[n]["h"][L][torch.as_tensor(f.fi.values)] for n, f in train_sets])
    ytr = torch.tensor(np.concatenate([f.target.values for _, f in train_sets]))
    for tag, ft in tests.items():
        src = "ami" if tag.startswith("ami") else "librispeech"
        m, lo, hi = mean_ci(word_hit(None, TL_FRAMES[src]["h"][L], ft))
        tl_rows.append(dict(layer=L, lens="logit", rank=0, test=tag, hit=m, ci_lo=lo, ci_hi=hi))
    for rank in CFG["tl_ranks"]:
        lens = train_lens(Htr, ytr, CFG["tl_steps"], CFG["seed"], rank)
        shuf = train_lens(Htr, ytr, CFG["tl_steps"], CFG["seed"], rank, shuffle=True)
        for tag, ft in tests.items():
            src = "ami" if tag.startswith("ami") else "librispeech"
            for lname, ln in (("tuned", lens), ("shuffled", shuf)):
                m, lo, hi = mean_ci(word_hit(ln, TL_FRAMES[src]["h"][L], ft))
                tl_rows.append(dict(layer=L, lens=lname, rank=rank, test=tag, hit=m, ci_lo=lo, ci_hi=hi))
        del lens, shuf; torch.cuda.empty_cache()
    print(f"layer {L} done")
TL = pd.DataFrame(tl_rows); TL.to_csv(f"{CFG['out_dir']}/tuned_lens_v6.csv", index=False)
for tag in tests:
    print(f"--- {tag}"); print(TL[TL.test == tag].pivot_table(index="layer", columns=["lens", "rank"], values="hit").round(3).to_string())

fig, axes = plt.subplots(1, len(tests), figsize=(6 * len(tests), 4), squeeze=False)
for ax, tag in zip(axes[0], tests):
    for (ln, rk), g in TL[TL.test == tag].groupby(["lens", "rank"]):
        lab = "logit lens" if ln == "logit" else f"{ln} r={rk}"
        ax.plot(g.layer, g.hit, "-o", ms=3, label=lab, ls=":" if ln == "shuffled" else "-")
        ax.fill_between(g.layer, g.ci_lo, g.ci_hi, alpha=.12)
    ax.set(title=f"{CFG['model']}: {tag} words", xlabel="layer", ylabel="word hit@5"); ax.grid(alpha=.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{CFG['out_dir']}/tuned_lens_v6.png", dpi=150); plt.show()

## 11. Сохранение

In [ ]:
for (name, cond), turns in RUNS.items():
    if name not in MAIN: continue
    d = f"{CFG['out_dir']}/{name}_{cond}"
    os.makedirs(d, exist_ok=True)
    np.savez_compressed(f"{d}/top5.npz", **{t["id"]: t["top"][:, :, :5].numpy().astype(np.int32) for t in turns})
    json.dump([dict(id=t["id"], spk=t["spk"], user_start=t["user_start"], n_user=t["n_user"], text=t["text"], words=t["words"],
                    own_words=t["own_words"], own_interval=list(t["own_interval"])) for t in turns], open(f"{d}/turns.json", "w"))
    json.dump(dict(CFG, model=f"{CFG['model']}|{name}|{cond}"), open(f"{d}/config.json", "w"), indent=2)
json.dump(dict(CFG, aligner_used=ALIGNER_USED), open(f"{CFG['out_dir']}/config.json", "w"), indent=2)
import shutil
from IPython.display import FileLink, display
archive = shutil.make_archive(CFG["out_dir"], "zip", CFG["out_dir"])
print(archive, f"{os.path.getsize(archive) / 1e6:.1f} MB")
os.chdir(BASE); display(FileLink(os.path.basename(archive)))

## Как читать
- **H10:** если `own_reversed` и `own_noise` подавляют так же, как речь, — дело в активности своего канала; если слабее — в словах своей речи. `own_quiet` показывает, зависит ли эффект от громкости.
- **H11:** пунктирные кривые: читает ли модель слова собственного канала. Ноль при сильном подавлении собеседника = «не слушает ни себя, ни его».
- **H9a:** главное — `log-ratio of P(heard words)` < 0 с интервалом ниже нуля и `KL own-ctrl` > 0: без транскрипции ответ «забывает» услышанное.
- **H9b:** `own-ctrl` < 0 для слов второй половины (особенно предсказуемых) при малом перекрытии подпространств: транскрипция ранних слов служит контекстом для узнавания следующих.
- **H5″:** `unseen_target` у tuned lens выше logit lens и перемешанного контроля со слоя ~12.